<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/geospatial/HW_6/A5_L8_GeoAnomaly_Mazibuko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Geo-anomaly detection** - Kwanda Mazibuko - stdnr: 1077167

# **0. Load Libraries**

In [27]:
!pip -q install yellowbrick geostatspy statsmodels geostatsmodels
!pip -q install numba plotly scikit-learn tabulate

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import kurtosis
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable

import matplotlib as mpl
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

from sklearn.preprocessing import MinMaxScaler

from sklearn.impute import KNNImputer
from sklearn.metrics import r2_score

from statsmodels.stats.weightstats import DescrStatsW         # for weighted statistics calculations
from scipy.optimize import curve_fit                          # for curve fitting
import geostatspy.GSLIB as GSLIB                              # GSLIB utilities, visualization, and wrapper functions
import geostatspy.geostats as geostats                        # GSLIB methods converted to Python
import geostatspy                                             # for accessing the package version and general utilities
#print('GeostatsPy version: ' + str(geostatspy.__version__))   # print the version of GeostatsPy

from tqdm import tqdm                                         # progress bar for loops
from functools import partialmethod                           # used to modify tqdm to suppress the status bar
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)    # suppress the status bar for tqdm

from matplotlib.ticker import (MultipleLocator, AutoMinorLocator) # control over axes ticks in plots
plt.rc('axes', axisbelow=True)                                # ensure grid lines are plotted below plot elements

import scipy.spatial as sp                                    # for spatial data structures and algorithms (KDTree)
import geopandas as gpd                                       # for handling geospatial data (GeoDataFrame)
from matplotlib import gridspec                               # custom subplots

# Define ignore_warnings before using it
ignore_warnings = True                                        # flag to ignore warnings

# Check and apply the warning filter
if ignore_warnings == True:
    import warnings
    warnings.filterwarnings('ignore')

from IPython.utils import io                                  # mute output from simulation

cmap = "Spectral_r"


#ignoring warnings
import warnings
warnings.filterwarnings('ignore')

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# **1. Data Loading & Inspection**

In [ ]:
#load data
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/geospatial/HW_6/Processed_Assen_RS_data.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
df.shape

In [ ]:
# col names
coord_names = ['Longitude', 'Latitude']
bands = ['Band1', 'Band2','Band3', 'Band4', 'Band5', 'Band6', 'Band7']

#### **Summary Statistics**

In [ ]:
# Summary Stat
df.describe().T

#### **Missingness**

In [ ]:
# Missingness
# Rows and Columns of data
print('Data has {} rows and {} Columns'.format(df.shape[0],df.shape[1]))
print('')
print('Data has the following variables:')
print('')
for i in df.columns:
  missing_values = 100*df[i].isnull().sum()/df.shape[0]
  print(i+', with {}% of missing data'.format(missing_values))
  print('')

- No column with missing values.

#### **Kurtosis for Outlier Detection**

In [ ]:
df_temp = df.copy()
features = bands

# Calculate kurtosis for each feature
kurtosis_values = {
    'Feature': [],
    'Kurtosis': [],
    'Interpretation': [],
    'Kurtosis Value Interpretation': []
}

k_meaning = '<3: Light tails (fewer outliers), ~3: Normal distribution, >3: Heavy tails (more outliers)'

for col in features.columns:
    k = kurtosis(df_temp[col], fisher=True, bias=False)
    kurtosis_values['Feature'].append(col)
    kurtosis_values['Kurtosis'].append(round(k, 3))
    kurtosis_values['Kurtosis Value Interpretation'].append(k_meaning)
    if k > 3:
        interpretation = "Heavy tails (many outliers)"
    elif k < 3:
        interpretation = "Light tails (few outliers)"
    else:
        interpretation = "Normal-like tails"
    kurtosis_values['Interpretation'].append(interpretation)

# Display the results
kurtosis_df = pd.DataFrame(kurtosis_values)
print(tabulate(kurtosis_df, headers = 'keys', tablefmt = 'grid'))

# **2. Spatial Exploratory Data Analysis**

In [ ]:
# spatial Maps of all bands
for Band in bands:

        fig = plt.figure(figsize = (10,8))
        ax = fig.add_subplot(111)

        ax.set_aspect('equal')
        ax.set_xlabel('Longitude', fontsize = 10)
        ax.set_ylabel('Latitude', fontsize = 10)
        ax.set_title(Band+' Landsat 8', fontsize = 10)

        colormap = plt.cm.get_cmap('jet',int((max(df[Band])-min(df[Band]))*20))

        scatter=ax.scatter(df.Longitude, df.Latitude, c = df[Band],
                           cmap = colormap, s = 3,alpha = 1.0, edgecolors = 'none', label = 'Landsat 8')

        cbar = fig.colorbar(scatter, pad = -0.000)
        cbar.set_label(Band)
        ax.grid(True)
        plt.show()


#### **corelation heatmap**

In [ ]:
# correlation heatmap
corr_matrix = df[bands].copy().corr()

plt.figure(figsize = (8, 6))
sns.heatmap(corr_matrix, annot = True, cmap = 'coolwarm', fmt = ".2f")
plt.title('Correlation Heatmap')
plt.show()


**Observations**

- High correlations (>= 0.85) dominate the matrix, especially among Bands 1 to 4, 6, and 7.

- Band5 stands out with consistently low correlations, indicating it captures different spectral information.

Bands 1–4, 6, and 7
These bands are strongly interrelated, with correlation coefficients mostly above 0.85.
- Band1 & Band2: 0.96
- Band3 & Band4: 0.96
- Band6 & Band7: 0.90

This means that these bands they may be capturing similar features and possibly overlapping spectral ranges or redundant information.

Band5
Shows weak correlation with all other bands:
- Band1: 0.21
- Band2: 0.33
- Band4: 0.37
- Band6: 0.50

This means that Band5 is likely measuring a distinct spectral region or feature, making it valuable for complementary analysis or dimensionality reduction.

# **3. Data pre-processing & scaling**

# **4. Regression to reconstruct B4, B5, B6**

# **5. Residual analysis (per band)**

# **6. PCA on residuals (Bands 4, 5, 6)**